In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

from nd_aligner.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from nd_aligner.config.utils.io import load_config
from nd_aligner.models.ndaligner import init_nd_aligner_training_module

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [2]:
aligner_training_module_cfg_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/main/nd_aligner_vctk+libritts_test_20260829-134700/model_config.json"
aligner_training_module_ckpt_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/main/nd_aligner_vctk+libritts_test_20260829-134700/checkpoints_timit_bae/best_step_timit_bae_0.020939_step_134000_epoch_5.pth"

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# aligner_training_module_cfg_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/TCD_rf_sweep/nd_aligner_vctk_20260821-215400/model_config.json"
# aligner_training_module_ckpt_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/TCD_rf_sweep/nd_aligner_vctk_20260821-215400/checkpoints_timit_bae/best_step_timit_bae_0.020831_step_139000_epoch_38.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in /shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/main/nd_aligner_vctk+libritts_test_20260829-134700/checkpoints_timit_bae/best_step_timit_bae_0.020939_step_134000_epoch_5.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [4]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [5]:
from nd_aligner.benchmark.timit.benchmarker import TIMITBenchMarker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
assert aligner.input_maker is not None

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
    boundary_mode="both",
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [6]:
with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        max_test_samples=None,
    )


Computing Alignments: 100%|██████████| 1680/1680 [00:59<00:00, 28.01it/s]


In [7]:
print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")
print(f"{metrics.coverage_ratio * 100.0:.2f} %")

20.35 ms
98.54 %
89.38 %
73.54 %
45.18 %
98.02 %


## INIT BenchMarkers (Buckeye)

In [8]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [9]:
from nd_aligner.benchmark.timit.benchmarker import TIMITBenchMarker
assert aligner.input_maker is not None

# go here and run this: ./benchmark/buckeye/buckeye_to_timit_eval_format.py
BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [12]:
with torch.inference_mode():
    metrics = buckeye_benchmarker.__call__(
        aligner=aligner,
        max_test_samples=None,
    )

Computing Alignments:   0%|          | 0/19273 [00:00<?, ?it/s]

Computing Alignments: 100%|██████████| 19273/19273 [12:15<00:00, 26.22it/s]


In [13]:
print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")
print(f"{metrics.coverage_ratio * 100.0:.2f} %")

21.86 ms
97.36 %
90.57 %
74.57 %
43.59 %
98.17 %
